# DisasterSeverity — Targeted Fix to 0.82+ (BanglaBERT base only path)

## Root-cause diagnosis on the 0.784 baseline
```
Label         Expected in test   Actually predicted   Ratio
Catastrophic        ~57               161             2.81×  ← BROKEN
Moderate           ~239               165             0.69×  ← BROKEN
Minimal (non-ND)    ~36                 5             0.14×  ← BROKEN
```
The 2.75× class weight for Catastrophic caused the model to predict it **104 extra times**.
Assuming half are wrong, that's ~52 / 790 = 6.6 % error rate from one miscalibration alone.

## Three fixes (no large models used)
| Fix | What it does | Expected gain |
|-----|-------------|---------------|
| √ class weights | Catastrophic weight: 2.75× → 1.66× | +2–3 % |
| OOF temperature calibration | Auto-correct residual over/under-prediction | +0.5–1 % |
| `xlm-roberta-base` ensemble | Base-size (125M) different arch → diverse errors | +1–2 % |
| AWP replacing FGM | Perturbs all weights, not just embeddings | +0.3–0.7 % |
| LLRD | Lower LR for early layers, prevents forgetting | +0.3–0.5 % |

In [1]:
# ── Cell 1: Imports & Seed ─────────────────────────────────────────────────
import os, re, random, warnings, zipfile
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from scipy.special import softmax as scipy_softmax
from scipy.optimize import minimize_scalar
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, set_seed,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [2]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
BASE_PATH = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"

# ── FIX #1: Softer class weights via sqrt ─────────────────────────────────
# Old formula : N / (5 * count)          → Catastrophic gets 2.75×
# New formula : sqrt(N / (5 * count))    → Catastrophic gets 1.66×
# The sqrt dampens the minority-class boost so the model isn't pressured
# into predicting Catastrophic 3× more often than it actually appears.
WEIGHT_MODE = "sqrt"     # "standard" | "sqrt" | "none"

# ── Two BASE-sized models (neither is 'large') ─────────────────────────────
# BanglaBERT : 110M ELECTRA-style, Bangla-specific
# XLM-R base : 125M RoBERTa-style, multilingual — different arch → diverse errors
MODEL_CFGS = [
    dict(key="banglabert",  model_name="csebuetnlp/banglabert",  lr=2e-5, epochs=5),
    dict(key="xlmr_base",   model_name="xlm-roberta-base",       lr=2e-5, epochs=5),
]

# ── Shared hypers (unchanged from v2 unless noted) ─────────────────────────
CFG = dict(
    max_len         = 128,
    batch_size      = 16,
    weight_decay    = 0.01,
    warmup_ratio    = 0.10,
    label_smoothing = 0.10,
    n_folds         = 5,
    seed            = 42,
    # R-Drop
    use_rdrop       = True,
    rdrop_alpha     = 0.50,
    # AWP (replaces FGM — perturbs ALL encoder weights, not just embeddings)
    use_awp         = True,
    awp_lr          = 1e-4,
    awp_eps         = 1e-3,
    # LLRD
    use_llrd        = True,
    llrd_decay      = 0.90,
    # Pseudo-labelling
    pseudo_threshold = 0.85,   # was 0.90 — looser to get more samples
    pseudo_epochs    = 3,
    pseudo_lr        = 1e-5,
)

label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}
NUM_LABELS        = 5

In [3]:
# ── Cell 3: Data Loading ───────────────────────────────────────────────────
print("Loading data...")
train = pd.read_csv(f"{BASE_PATH}train.csv")
test  = pd.read_csv(f"{BASE_PATH}test.csv")
val   = pd.read_csv(f"{BASE_PATH}validation.csv")

train = pd.concat([train, val]).reset_index(drop=True)

# Category-aware input (kept from v2)
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

train["label_id"] = train["label"].map(label_map)

# Stratified 5-fold
skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
train["fold"] = -1
for fold, (_, vi) in enumerate(skf.split(train, train["label_id"])):
    train.loc[vi, "fold"] = fold

# ── FIX #1: Compute class weights with sqrt dampening ─────────────────────
raw_counts = np.bincount(train["label_id"].values, minlength=NUM_LABELS).astype(float)
raw_w      = len(train) / (NUM_LABELS * raw_counts)   # standard formula

if WEIGHT_MODE == "sqrt":
    CLASS_WEIGHTS = np.sqrt(raw_w).tolist()
elif WEIGHT_MODE == "none":
    CLASS_WEIGHTS = [1.0] * NUM_LABELS
else:
    CLASS_WEIGHTS = raw_w.tolist()

print("Class weights (standard vs sqrt):")
for label, s, sq in zip(label_map, raw_w, CLASS_WEIGHTS):
    print(f"  {label:12s}: standard={s:.3f}  sqrt={sq:.3f}")
print(f"\nTrain: {len(train)} | Test: {len(test)}")

Loading data...
Class weights (standard vs sqrt):
  Minimal     : standard=1.171  sqrt=1.082
  Mild        : standard=0.985  sqrt=0.992
  Moderate    : standard=0.659  sqrt=0.812
  Severe      : standard=0.800  sqrt=0.894
  Catastrophic: standard=2.751  sqrt=1.659

Train: 3590 | Test: 790


In [4]:
# ── Cell 4: AWP · LLRD · AdvancedTrainer ──────────────────────────────────

# ── AWP: Adversarial Weight Perturbation ───────────────────────────────────
# Stronger than FGM: perturbs ALL encoder weights (not just embeddings)
# and clamps each perturbation inside an ε-ball of the original value.
SKIP_AWP = ("bias", "LayerNorm", "layer_norm", "classifier", "pooler")

class AWP:
    def __init__(self, model):
        self.model  = model
        self.backup = {}

    def attack(self, adv_lr, adv_eps):
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.grad is None:
                continue
            if any(s in name for s in SKIP_AWP):
                continue
            self.backup[name] = param.data.clone()
            norm = torch.norm(param.grad)
            if norm != 0 and not torch.isnan(norm):
                r = adv_lr * param.grad / norm
                param.data.add_(r)
                param.data = torch.clamp(
                    param.data,
                    self.backup[name] - adv_eps,
                    self.backup[name] + adv_eps,
                )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


# ── LLRD: Layer-wise Learning Rate Decay ───────────────────────────────────
def get_llrd_params(model, base_lr, weight_decay, decay=0.90):
    """
    Each transformer layer gets LR × decay^(depth_from_top).
    Embeddings get the lowest LR, classifier head gets base_lr.
    Works for both BanglaBERT (ELECTRA) and XLM-R (RoBERTa).
    """
    no_decay  = ("bias", "LayerNorm.weight", "LayerNorm.bias",
                 "layer_norm.weight", "layer_norm.bias")
    n_layers  = getattr(model.config, "num_hidden_layers", 12)

    def depth(name):
        if any(h in name for h in ("classifier", "pooler", "discriminator_predictions")):
            return n_layers + 1
        if "embeddings" in name:
            return 0
        m = re.search(r"\.layer\.(\d+)\.", name)
        return int(m.group(1)) + 1 if m else n_layers

    groups = defaultdict(lambda: {"decay": [], "no_decay": []})
    for name, param in model.named_parameters():
        if param.requires_grad:
            d   = depth(name)
            key = "no_decay" if any(nd in name for nd in no_decay) else "decay"
            groups[d][key].append(param)

    max_d = n_layers + 1
    param_groups = []
    for d, ps in groups.items():
        lr = base_lr * (decay ** (max_d - d))
        if ps["decay"]:
            param_groups.append({"params": ps["decay"],    "lr": lr, "weight_decay": weight_decay})
        if ps["no_decay"]:
            param_groups.append({"params": ps["no_decay"], "lr": lr, "weight_decay": 0.0})
    return param_groups


# ── Metrics ────────────────────────────────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds), "f1": f1}


# ── Advanced Trainer ───────────────────────────────────────────────────────
class AdvancedTrainer(Trainer):
    """
    Changes vs v2 baseline:
      • sqrt class weights (FIX #1 — less Catastrophic over-prediction)
      • AWP replacing FGM (stronger adversarial)
      • LLRD via create_optimizer override
      • R-Drop + label smoothing unchanged
    """

    def _ce(self, logits, labels):
        wt = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(logits.device)
        return nn.CrossEntropyLoss(weight=wt, label_smoothing=CFG["label_smoothing"])(
            logits, labels
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        # R-Drop only during training; disabled during AWP pass (_awp_mode)
        if model.training and CFG["use_rdrop"] and not getattr(self, "_awp_mode", False):
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce   = (self._ce(out1.logits, labels) + self._ce(out2.logits, labels)) / 2
            p1, p2 = F.softmax(out1.logits, -1), F.softmax(out2.logits, -1)
            kl = (F.kl_div(out1.logits.log_softmax(-1), p2, reduction="batchmean") +
                  F.kl_div(out2.logits.log_softmax(-1), p1, reduction="batchmean")) / 2
            loss = ce + CFG["rdrop_alpha"] * kl
            return (loss, out1) if return_outputs else loss
        else:
            out  = model(**inputs)
            loss = self._ce(out.logits, labels)
            return (loss, out) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        inputs = self._prepare_inputs(inputs)
        labels = inputs["labels"].clone()

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        scale = self.args.gradient_accumulation_steps
        self.accelerator.backward(loss / scale if scale > 1 else loss)

        if CFG["use_awp"]:
            awp = AWP(model)
            awp.attack(CFG["awp_lr"], CFG["awp_eps"])
            inputs["labels"] = labels
            self._awp_mode = True
            with self.compute_loss_context_manager():
                loss_adv = self.compute_loss(model, inputs)
            self._awp_mode = False
            self.accelerator.backward(loss_adv / scale if scale > 1 else loss_adv)
            awp.restore()

        return loss.detach()

    def create_optimizer(self):
        if CFG["use_llrd"]:
            grouped = get_llrd_params(
                self.model, self.args.learning_rate,
                self.args.weight_decay, CFG["llrd_decay"]
            )
            self.optimizer = torch.optim.AdamW(grouped, eps=1e-6)
        else:
            self.optimizer = super().create_optimizer()
        return self.optimizer

In [5]:
# ── Cell 5: Generic Training Function (collects OOF logits for calibration) ─
def train_model(model_cfg):
    """
    5-fold CV for one model config.
    Returns:
      test_logits : mean of fold test logits  (n_test, 5)
      oof_logits  : out-of-fold logits        (n_train, 5)  ← used for calibration
      oof_labels  : matching true labels      (n_train,)
      mean_cv_f1  : float
    """
    key       = model_cfg["key"]
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_name"])

    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length",
                         truncation=True, max_length=CFG["max_len"])

    tok_test = Dataset.from_pandas(test[["text"]]).map(tokenize_fn, batched=True)

    fold_test_logits = []
    cv_f1s           = []
    oof_logits_list  = [None] * len(train)  # will fill in per-fold
    oof_labels_arr   = train["label_id"].values.copy()

    for fold in range(CFG["n_folds"]):
        print(f"\n{'='*18} [{key.upper()}] FOLD {fold+1}/{CFG['n_folds']} {'='*18}")
        seed_everything(CFG["seed"] + fold)

        trn_df = train[train["fold"] != fold].reset_index(drop=True)
        val_df = train[train["fold"] == fold].reset_index(drop=True)
        val_idx = train[train["fold"] == fold].index.tolist()

        tok_trn = (Dataset.from_pandas(
            trn_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))
        tok_val = (Dataset.from_pandas(
            val_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))

        model = AutoModelForSequenceClassification.from_pretrained(
            model_cfg["model_name"], num_labels=NUM_LABELS
        )

        args = TrainingArguments(
            output_dir                  = f"/kaggle/working/{key}_fold{fold}",
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            learning_rate               = model_cfg["lr"],
            per_device_train_batch_size = CFG["batch_size"],
            per_device_eval_batch_size  = CFG["batch_size"],
            num_train_epochs            = model_cfg["epochs"],
            warmup_ratio                = CFG["warmup_ratio"],
            lr_scheduler_type           = "cosine",
            weight_decay                = CFG["weight_decay"],
            load_best_model_at_end      = True,
            metric_for_best_model       = "f1",
            greater_is_better           = True,
            report_to                   = "none",
            save_total_limit            = 1,
        )

        trainer = AdvancedTrainer(
            model=model, args=args,
            train_dataset=tok_trn, eval_dataset=tok_val,
            compute_metrics=compute_metrics,
        )
        trainer.train()

        best_f1 = max(
            (x.get("eval_f1", -1) for x in trainer.state.log_history), default=0
        )
        cv_f1s.append(best_f1)
        print(f"Fold {fold+1} best F1: {best_f1:.4f}")

        # ── Collect OOF logits (for calibration in Cell 7) ──────────────
        oof_preds = trainer.predict(tok_val)
        for i, orig_idx in enumerate(val_idx):
            oof_logits_list[orig_idx] = oof_preds.predictions[i]

        # ── Test prediction ──────────────────────────────────────────────
        fold_test_logits.append(trainer.predict(tok_test).predictions)

        # ── CLEANUP: Free RAM, VRAM, and Disk Space ──────────────────────
        out_dir = f"/kaggle/working/{key}_fold{fold}"
        
        del model, trainer
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        
        if os.path.exists(out_dir):
            import shutil
            shutil.rmtree(out_dir, ignore_errors=True)

    mean_cv = np.mean(cv_f1s)
    print(f"\n[{key}] Mean CV F1: {mean_cv:.4f}  {[round(s,4) for s in cv_f1s]}")

    # Save for potential resume
    np.save(f"/kaggle/working/{key}_test_logits.npy",  np.array(fold_test_logits))
    np.save(f"/kaggle/working/{key}_oof_logits.npy",   np.array(oof_logits_list))

    return (
        np.mean(fold_test_logits, axis=0),   # (n_test,  5)
        np.array(oof_logits_list),            # (n_train, 5)
        oof_labels_arr,
        mean_cv,
    )

In [6]:
# ── Cell 6: Train all models ───────────────────────────────────────────────
results = {}   # key → {test_logits, oof_logits, oof_labels, cv_f1}

for mcfg in MODEL_CFGS:
    seed_everything(CFG["seed"])
    tl, oof_l, oof_y, cv = train_model(mcfg)
    results[mcfg["key"]] = dict(
        test_logits = tl,
        oof_logits  = oof_l,
        oof_labels  = oof_y,
        cv_f1       = cv,
    )

print("\n── CV Summary ──")
for k, v in results.items():
    print(f"  {k}: {v['cv_f1']:.4f}")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/790 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Fold 5 best F1: 0.5862



[xlmr_base] Mean CV F1: 0.5951  [0.5632, 0.6067, 0.6076, 0.6115, 0.5862]

── CV Summary ──
  banglabert: 0.6201
  xlmr_base: 0.5951


In [7]:
# ── Cell 7: OOF Temperature Calibration (FIX #2) ──────────────────────────
#
# Even after sqrt class weights, there may still be residual miscalibration.
# Temperature scaling finds a scalar T on out-of-fold predictions such that
# logits / T maximises weighted F1 on the validation data.
# The SAME T is then applied to test logits before the ensemble.
#
# T > 1 → softer probabilities (less confident) → reduces over-prediction
# T < 1 → sharper probabilities (more confident)
# ──────────────────────────────────────────────────────────────────────────
def find_temperature(oof_logits, oof_labels):
    def neg_f1(T):
        preds = scipy_softmax(oof_logits / T, axis=-1).argmax(axis=-1)
        _, _, f1, _ = precision_recall_fscore_support(
            oof_labels, preds, average="weighted", zero_division=0
        )
        return -f1
    result = minimize_scalar(neg_f1, bounds=(0.3, 4.0), method="bounded")
    return result.x


temperatures = {}
print("Calibration temperatures (T > 1 = was over-confident):")
for key, res in results.items():
    T = find_temperature(res["oof_logits"], res["oof_labels"])
    temperatures[key] = T

    # Verify improvement on OOF data
    raw_preds  = res["oof_logits"].argmax(axis=-1)
    cal_preds  = scipy_softmax(res["oof_logits"] / T, axis=-1).argmax(axis=-1)
    _, _, f1_raw, _ = precision_recall_fscore_support(
        res["oof_labels"], raw_preds, average="weighted", zero_division=0
    )
    _, _, f1_cal, _ = precision_recall_fscore_support(
        res["oof_labels"], cal_preds, average="weighted", zero_division=0
    )
    print(f"  {key}: T={T:.3f}  OOF F1 raw={f1_raw:.4f} → calibrated={f1_cal:.4f}  Δ={f1_cal-f1_raw:+.4f}")

# Apply calibration to test logits
for key in results:
    T = temperatures[key]
    results[key]["test_logits_cal"] = results[key]["test_logits"] / T

Calibration temperatures (T > 1 = was over-confident):
  banglabert: T=4.000  OOF F1 raw=0.6207 → calibrated=0.6207  Δ=+0.0000
  xlmr_base: T=4.000  OOF F1 raw=0.5951 → calibrated=0.5951  Δ=+0.0000


In [8]:
# ── Cell 8: Score-Weighted Ensemble ───────────────────────────────────────
total_cv = sum(v["cv_f1"] for v in results.values())
weights  = {k: v["cv_f1"] / total_cv for k, v in results.items()}

print("Ensemble weights (by CV F1):")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k}: {w:.3f}  (CV F1 = {results[k]['cv_f1']:.4f})")

# Weighted sum of calibrated test logits
ensemble_logits = sum(
    weights[k] * results[k]["test_logits_cal"]
    for k in results
)

final_preds   = np.argmax(ensemble_logits, axis=-1)
test["label"] = [reverse_label_map[p] for p in final_preds]

# Hard rule: Non Disaster always Minimal
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"

# ── Advanced Hard Rules ──
test.loc[(test["category"] == "Tropical Storm") & (test["label"] == "Catastrophic"), "label"] = "Severe"
test.loc[(test["category"] == "Human Damage") & (test["label"] == "Catastrophic"), "label"] = "Severe"
test.loc[(test["category"] == "Flood") & (test["label"] == "Minimal"), "label"] = "Mild"

print("\nPrediction distribution (compare with v2 to see Catastrophic drop):")
print(test["label"].value_counts())
print()
# Sanity check: expected vs predicted ratio for Catastrophic
n_non_nd = (test["category"] != "Non Disaster").sum()
n_cat    = (test["label"] == "Catastrophic").sum()
print(f"Catastrophic rate in test (non-ND): {n_cat}/{n_non_nd} = {100*n_cat/n_non_nd:.1f}%")
print(f"Expected from training dist:  8.3%  |  v2 predicted: 23.3%")

Ensemble weights (by CV F1):
  banglabert: 0.510  (CV F1 = 0.6201)
  xlmr_base: 0.490  (CV F1 = 0.5951)

Prediction distribution (compare with v2 to see Catastrophic drop):
label
Moderate        242
Severe          230
Mild            127
Minimal         109
Catastrophic     82
Name: count, dtype: int64

Catastrophic rate in test (non-ND): 82/690 = 11.9%
Expected from training dist:  8.3%  |  v2 predicted: 23.3%


In [9]:
# ── Cell 9: Pseudo-Labelling ───────────────────────────────────────────────
# Threshold lowered to 0.85 (was 0.90) to get more pseudo samples.
# Uses the CALIBRATED ensemble logits so pseudo-labels are better quality.
print("── Pseudo-Labelling ──")
probs     = scipy_softmax(ensemble_logits, axis=-1)
max_probs = probs.max(axis=-1)
confident = max_probs >= CFG["pseudo_threshold"]
print(f"Confident test samples (≥{CFG['pseudo_threshold']}): {confident.sum()}/{len(test)}")

if confident.sum() > 50:
    pseudo_df              = test[confident].copy()
    pseudo_df["label_id"]  = final_preds[confident]

    full_train = pd.concat(
        [train[["text", "label_id"]], pseudo_df[["text", "label_id"]]],
        ignore_index=True,
    )
    print(f"Expanded train: {len(train)} → {len(full_train)} samples")

    # Use the highest-CV model for pseudo training
    best_key = max(results, key=lambda k: results[k]["cv_f1"])
    best_cfg = next(c for c in MODEL_CFGS if c["key"] == best_key)
    tokenizer_p = AutoTokenizer.from_pretrained(best_cfg["model_name"])

    def tok_p(examples):
        return tokenizer_p(examples["text"], padding="max_length",
                           truncation=True, max_length=CFG["max_len"])

    tok_full = (
        Dataset.from_pandas(full_train.rename(columns={"label_id": "label"}))
        .map(tok_p, batched=True)
    )
    tok_tst = Dataset.from_pandas(test[["text"]]).map(tok_p, batched=True)

    pseudo_model = AutoModelForSequenceClassification.from_pretrained(
        best_cfg["model_name"], num_labels=NUM_LABELS
    )
    pseudo_args = TrainingArguments(
        output_dir                  = "/kaggle/working/pseudo",
        num_train_epochs            = CFG["pseudo_epochs"],
        per_device_train_batch_size = CFG["batch_size"],
        per_device_eval_batch_size  = CFG["batch_size"],
        learning_rate               = CFG["pseudo_lr"],
        warmup_ratio                = CFG["warmup_ratio"],
        lr_scheduler_type           = "cosine",
        weight_decay                = CFG["weight_decay"],
        save_strategy               = "no",
        report_to                   = "none",
    )
    pseudo_trainer = AdvancedTrainer(
        model=pseudo_model, args=pseudo_args,
        train_dataset=tok_full, compute_metrics=compute_metrics,
    )
    pseudo_trainer.train()
    pseudo_logits = pseudo_trainer.predict(tok_tst).predictions

    # Calibrate pseudo logits with the best-model temperature
    pseudo_logits_cal = pseudo_logits / temperatures[best_key]

    # Blend: 70% ensemble + 30% pseudo
    blended      = 0.70 * ensemble_logits + 0.30 * pseudo_logits_cal
    final_preds  = np.argmax(blended, axis=-1)
    test["label"] = [reverse_label_map[p] for p in final_preds]
    test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
    
# ── Advanced Hard Rules ──
    test.loc[(test["category"] == "Tropical Storm") & (test["label"] == "Catastrophic"), "label"] = "Severe"
    test.loc[(test["category"] == "Human Damage") & (test["label"] == "Catastrophic"), "label"] = "Severe"
    test.loc[(test["category"] == "Flood") & (test["label"] == "Minimal"), "label"] = "Mild"
    print("Pseudo-labelling complete. Final distribution:")
    print(test["label"].value_counts())
else:
    print("Too few confident samples; skipping pseudo-labelling.")

── Pseudo-Labelling ──
Confident test samples (≥0.85): 0/790
Too few confident samples; skipping pseudo-labelling.


In [10]:
# ── Cell 10: Save Submission ───────────────────────────────────────────────
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")

print("✅ submission.zip ready.")
print(submission["label"].value_counts())
print(submission.head(10))

✅ submission.zip ready.
label
Moderate        242
Severe          230
Mild            127
Minimal         109
Catastrophic     82
Name: count, dtype: int64
         image_id     label
0  landslides_804    Severe
1  landslides_805  Moderate
2  landslides_806      Mild
3  landslides_807      Mild
4  landslides_808  Moderate
5  landslides_809  Moderate
6  landslides_810  Moderate
7  landslides_811  Moderate
8  landslides_812      Mild
9  landslides_813    Severe


## What each fix does to your specific 0.784 run

```
v2 prediction:  Catastrophic=161 (23.3% of non-ND) ← model is broken here
Expected:        Catastrophic= 57 ( 8.3% of non-ND)
Excess wrong:   ~100 samples  → costs ~3% weighted F1

Fix 1  √ class weights  → Catastrophic weight 2.75→1.66  → model stops over-firing
Fix 2  Temperature T    → found automatically on OOF data → residual correction
Fix 3  XLM-R base       → different architecture, different error pattern
                         → ensemble averages out the individual mistakes
```

## Diagnostic: check Cell 8 output
After running, look at the `Catastrophic rate in test` printout:
- **8–12 %** → calibration worked, model is now reasonable  
- Still **>18 %** → lower `pseudo_threshold` to 0.80 and re-run pseudo step  

## If you still want one more point without large models
```python
# Add a third base model to MODEL_CFGS:
dict(key="muril", model_name="google/muril-base-cased", lr=2e-5, epochs=5)
# MuRIL is Google's multilingual model specifically covering
# Indian subcontinent languages (Bengali included), ~110M params.
```